In [1]:
# Install Conda
!pip install -q condacolab
import condacolab
condacolab.install()

# Mount Google Drive immediately
from google.colab import drive
drive.mount('/content/drive')

⏬ Downloading https://github.com/jaimergp/miniforge/releases/download/24.11.2-1_colab/Miniforge3-colab-24.11.2-1_colab-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:11
🔁 Restarting kernel...
Mounted at /content/drive


In [1]:
# 1. Download the code
!git clone https://github.com/varun-bhai/batch_peptide_pipeline.git
%cd batch_peptide_pipeline

# 2. Create the destination folder in your Google Drive
!mkdir -p /content/drive/MyDrive/finalo_randomoo

# 3. The Magic Trick: Link the local output folder directly to your Drive
!rm -rf /content/batch_peptide_pipeline/output
!ln -s /content/drive/MyDrive/finalo_randomoo /content/batch_peptide_pipeline/output

# 4. Build ETFlow Environment (Fixed Installation)
!conda create -n etflow_env python=3.10 -y

# Install basic packages FIRST so they don't get blocked
!conda run -n etflow_env pip install requests rdkit biopython

# Install a specific PyTorch version with pre-compiled wheels
!conda run -n etflow_env pip install torch==2.1.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

# Install torch-cluster from PyG's pre-built binary link (Prevents the build error)
!conda run -n etflow_env pip install torch-cluster==1.6.3 -f https://data.pyg.org/whl/torch-2.1.0+cu118.html

# Install ETFlow last
!conda run -n etflow_env pip install etflow

# 5. Build AlphaFold & OpenBabel
!conda create -n af2_env python=3.10 -y
!conda run -n af2_env pip install "colabfold[alphafold] @ git+https://github.com/sokrypton/ColabFold"
!conda install -c conda-forge openbabel -y

Cloning into 'batch_peptide_pipeline'...
remote: Enumerating objects: 31, done.
remote: Counting objects: 100% (31/31), done.
remote: Compressing objects: 100% (31/31), done.
remote: Total 31 (delta 10), reused 8 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (31/31), 80.96 KiB | 10.12 MiB/s, done.
Resolving deltas: 100% (10/10), done.
/content/batch_peptide_pipeline
Channels:
 - conda-forge
Platform: linux-64
Solving environment: / - done


==> WARNING: A newer version of conda exists. <==
    current version: 24.11.2
    latest version: 26.1.1

Please update conda by running

    $ conda update -n base -c conda-forge conda



## Package Plan ##

  environment location: /usr/local/envs/etflow_env

  added / updated specs:
    - python=3.10


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    _openmp_mutex-4.5          |           20_gnu          28 KB  conda-forge
    bzip2-

In [2]:
# Install biopython into the base Colab environment for Step 4
!pip install biopython

import os
os.environ['MPLBACKEND'] = 'Agg'

!python main.py --sequence "{nt:DiMe1}AKTA" --json modifications.json --job_name test_nt

!python main.py --sequence "MKTA{ct:0NC}" --json modifications.json --job_name test_ct

!python main.py --sequence "MK{nnr:0FL}TA" --json modifications.json --job_name test_nnr

!python main.py --sequence "MKA{ptm:bromo3}A" --json modifications.json --job_name test_ptm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 59.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.9/16.9 MB 57.2 MB/s eta 0:00:00



[INFO] Loaded 1 sequence from command line input

[INFO] Starting job 1/1: test_nt

[INFO] test_nt | Step 1/5: Parsing sequence
[INFO] Command: python /content/batch_peptide_pipeline/01_parse_input.py --sequence {nt:DiMe1}AKTA --json /content/batch_peptide_pipeline/modifications.json --fasta_out /content/batch_peptide_pipeline/output/test_nt/parsed_sequence.fasta --mods_out /content/batch_peptide_pipeline/output/test_nt/modifications.txt

[INFO] test_nt | Step 2/5: Backbone prediction (af2_env)
[INFO] Command: conda run --no-capture-output -n af2_env python /content/batch_peptide_pipeline/02_run_backbone.py --fasta /content/batch_peptide_pipeline/output/test_nt/parsed_sequence.fasta --out_pdb /content/batch_peptide_pipeline/output/test_nt/backbone.pdb

Submitting to ESMFold (attempt 1)...

[✓] ESMFold structure saved to /content/batch_peptide_pipeline/output/test_nt/backbone.pdb

[✓] Final backbone written to: /content/batch_peptide_pipeline/output/test_nt/backbone.pdb

[INFO] test_nt

In [ ]:
!python 01_parse_input.py --sequence "MKA{ptm:6CV}A" --json modifications.json --fasta_out ptm_test.fasta --mods_out ptm_test.txt
!echo "FASTA OUTPUT:"
!cat ptm_test.fasta
!echo -e "\nMODIFICATIONS LOG:"
!cat ptm_test.txt

In [ ]:
!python 01_parse_input.py --sequence "MKA{ptm:6CV}A" --json modifications.json --fasta_out ptm_test.fasta --mods_out ptm_test.txt
!echo "FASTA OUTPUT:"
!cat ptm_test.fasta
!echo -e "\nMODIFICATIONS LOG:"
!cat ptm_test.txt